# **Descripcion**

El dataset inicial fue incrementado con data augmentation utilizando flips, scales, rotations, blurs, etc. teniendo por cada imagen original tres copias modificadas. Luego se crearon los archivos .txt con los labels para cada imagen que requiere yolo para ser ejecutado.

**Detección de mates:**

Para una primera aproximación se utilizó yolov5s, que es la versión menos pesada y más rápida que nos ofrece yolov5. C
omo parametros se incluyeron 100 épocas, batch_size de 32 y tamaño de imagen 640. También se intentó con 200 épocas y con un batch_size de 16.

Los mejores resultados se obtuvieron con la primer opción
100 épocas,
batch_size de 32 y tamaño 640.

Luego se prosiguió a intentar con una mejor versión de yolo, yolov5m para lo cual tomamos la misma cantidad de épocas y un tamaño menor de imagen (dado que si no nos daba un error de out of memory) para lo cual se observaron resultados un poco mejores al anterior caso.

**Detección de material:**

Para la detección del material se utilizó una red similar, tomando como entrada los pesos obtenidos de la red que detectó a los objetos mate y teniendo las clases a detectar como "plastico", "metal", "madera" y "calabaza".

Para la ejecución de la detección se debe respetar el tamaño de las imagenes de las redes (416 en este caso) y utilizar los archivos de pesos correspondientes.


# Setup

Download repos and install requirements

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone repo
%cd yolov5
!pip install -r requirements.txt  # install

%cd ..
!git clone https://github.com/75-70-Robots/mate-detection
!ls

!pip install wandb

#Tests that we are using GPU
import torch
from IPython.display import Image, clear_output  # to display images

clear_output()
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Setup complete. Using torch 2.3.0+cu121 (Tesla T4)


# Data augmentation

Se declaran funciones que usan la lib imgaug para ampliar el dataset inicial y otras para transformar los .xml en .txt

In [ ]:
import numpy as np
import imgaug as ia
import imageio as imageio
import imgaug.augmenters as iaa
import os
import copy
from pathlib import Path
from imgaug.augmentables.bbs import BoundingBox, BoundingBoxesOnImage

def copy_and_load_bounding_boxes(imgObj, shape):
    boxes = []
    all_objs = copy.deepcopy(imgObj)
    for obj in all_objs:
        boxes.append(BoundingBox(x1 = obj["xmin"], x2 = obj["xmax"], y1 = obj["ymin"], y2 = obj["ymax"], label=obj["name"]))
    return BoundingBoxesOnImage(boxes, shape=shape)

# aug_id para agregarle al nombre de la img
def aug_image(img, aug_id = ""):
    image = imageio.imread(img["filename"])
    if image is None: print('Cannot find ', img["filename"])

    h, w, c = image.shape
    bbs = copy_and_load_bounding_boxes(img["object"], image.shape)

    seq = iaa.Sequential(
        [
            iaa.Fliplr(0.5), # horizontal flips
            iaa.Crop(percent=(0, 0.1)), # random crops
            # Small gaussian blur with random sigma between 0 and 0.5.
            # But we only blur about 50% of all images.
            iaa.Sometimes(
                0.5,
                iaa.GaussianBlur(sigma=(0, 0.5))
            ),
            # Strengthen or weaken the contrast in each image.
            iaa.LinearContrast((0.75, 1.5)),
            # Add gaussian noise.
            # For 50% of all images, we sample the noise once per pixel.
            # For the other 50% of all images, we sample the noise per pixel AND
            # channel. This can change the color (not only brightness) of the
            # pixels.
            iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.05*255), per_channel=0.5),
            # Make some images brighter and some darker.
            # In 20% of all cases, we sample the multiplier once per channel,
            # which can end up changing the color of the images.
            iaa.Multiply((0.8, 1.2), per_channel=0.2),
            # Apply affine transformations to each image.
            # Scale/zoom them, translate/move them, rotate them and shear them.
            iaa.Affine(
                scale=(0.8, 1.2),
                translate_percent={"x": (-0.2, 0.2), "y": (-0.2, 0.2)},
                rotate=(-5, 5),
                shear=(-8, 8)
            )
        ],
        random_order=True
    )

    aug_image, aug_bbs = seq(image=image, bounding_boxes=bbs)
    aug_bbs = aug_bbs.remove_out_of_image().clip_out_of_image()
    # aug_image = aug_bbs.draw_on_image(aug_image) Testing purpouses only

    newFileName = Path(img["filename"]).stem + "-aug" + str(aug_id)
    splitted_name = os.path.splitext(img["filename"])
    aug_image_name = splitted_name[0] + "-aug" + str(aug_id) + splitted_name[1]

    aug_img_obj = {'object':[],'filename': aug_image_name, 'width': w, 'height':h, 'labelfilename':labels_dir + newFileName + ".txt"}
    for aug_bb in aug_bbs.bounding_boxes:
        obj = {}
        xmin, ymin, xmax, ymax = aug_bb.x1_int, aug_bb.y1_int, aug_bb.x2_int, aug_bb.y2_int
        obj["xmin"] = xmin
        obj["xmax"] = xmax
        obj["ymin"] = ymin
        obj["ymax"] = ymax
        obj["name"] = aug_bb.label
        aug_img_obj["object"].append(obj)

    #print(aug_img_obj)
    imageio.imwrite(aug_image_name, aug_image)
    return aug_img_obj

# Esta función crea archivos .txt que soporta yolov5 en base a los .XML
def process_annotations(ann_dir, img_dir, labels=[]):
    all_imgs = []

    for ann in [x for x in sorted(os.listdir(ann_dir)) if x.endswith('.xml')]:
        img = {'object':[]}
        img['labelfilename'] = ann_dir + ann
        tree = ET.parse(ann_dir + ann)

        for elem in tree.iter():
            if 'filename' in elem.tag:
                img['filename'] = img_dir + elem.text
            if 'width' in elem.tag:
                img['width'] = int(elem.text)
            if 'height' in elem.tag:
                img['height'] = int(elem.text)
            if 'object' in elem.tag or 'part' in elem.tag:
                obj = {}

                for attr in list(elem):
                    if 'name' in attr.tag:
                        obj['name'] = attr.text
                        if len(labels) > 0 and obj['name'] not in labels:
                            break
                        else:
                            img['object'] += [obj]

                    if 'bndbox' in attr.tag:
                        for dim in list(attr):
                            if 'xmin' in dim.tag:
                                obj['xmin'] = int(round(float(dim.text)))
                            if 'ymin' in dim.tag:
                                obj['ymin'] = int(round(float(dim.text)))
                            if 'xmax' in dim.tag:
                                obj['xmax'] = int(round(float(dim.text)))
                            if 'ymax' in dim.tag:
                                obj['ymax'] = int(round(float(dim.text)))

        if len(img['object']) > 0:
            all_imgs += [img]

    return all_imgs

def get_class_index(obj, labels):
    return labels.index(obj['name'])

def create_txt_from_xml_labels(train_imgs, labels, number_of_augmentations = 1):
    train_augmented_imgs = []
    for img in train_imgs:
        if not os.path.isfile(img["filename"]):
            continue
        train_augmented_imgs.append(img)
        #augment data
        #train_agumented_imgs.append(aug_image(img))
        for i in range(0, number_of_augmentations):
            try:
              train_augmented_imgs.append(aug_image(img, i))
            except:
              print("error in image: ",img)
            break

    for img in train_augmented_imgs:
        txtFilePath = os.path.splitext(img["labelfilename"])[0] + ".txt"
        imgWidth = img["width"]
        imgHeight = img["height"]
        with open(txtFilePath, 'w') as writer:
            for obj in img['object']:
                objClassIndex = get_class_index(obj, labels)
                if objClassIndex < 0:
                    continue
                boxWidth = (obj['xmax'] - obj['xmin']) / imgWidth
                boxHeight = (obj['ymax'] - obj['ymin']) / imgHeight
                xCenter = ((obj['xmax'] + obj['xmin']) / 2) / imgWidth
                yCenter = ((obj['ymax'] + obj['ymin']) / 2) / imgHeight
                line = str(objClassIndex) + " " + str(xCenter) + " " + str(yCenter) + " " + str(boxWidth) + " " + str(boxHeight)
                writer.write(line + os.linesep)

# Preprocessing mate labels
Preprocessing de labels para red de detección de mate unicamente.

In [ ]:
import os
import xml.etree.ElementTree as ET
%cd /content/yolov5
labels_dir = "../mate-detection/labels/mates/" # directorio que contiene los xml
img_dir = "../mate-detection/images/mates/"    # directorios con las imagenes
#labels = ['mate', 'plastico', 'calabaza', 'metal', 'madera']
labels = ['mate']

#aug_image(img_dir)

# Procesar .xml a objetos
train_imgs = process_annotations(labels_dir, img_dir, labels)
# Crear .txt para cada objeto
create_txt_from_xml_labels(train_imgs, labels, 3)

/content/yolov5


<ipython-input-2-8466cff65b04>:19: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(img["filename"])


In [ ]:
print(img_dir)

../mate-detection/images/mates/


# Create dataset yaml

Crear el .yaml usado por yolo para mate unicamente.
Se usa autosplit para dividir los datos en train/validation utilizando la regla 80/20

In [ ]:
import os

#autosplit(img_dir, (0.8, 0.2, 0))

with open("dataset.yaml", "w") as dataset:
    dataset.write("train: " + img_dir + os.linesep)
    dataset.write("val: " + img_dir + os.linesep)
    dataset.write("test: ../mate-detection/test-sets/test-set-01" + os.linesep)
    dataset.write("nc: " + str(len(labels)) + os.linesep)
    dataset.write("names: " + str(labels) + os.linesep)

# Train

Entrenamiento para detectar mate unicamente

In [ ]:
!python train.py --img 416 --batch 32 --epochs 100 --cfg yolov5s.yaml --data dataset.yaml --weights 'yolov5s.pt' --nosave --cache

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2024-06-05 00:06:01.155572: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-05 00:06:01.155623: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-05 00:06:01.209288: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: W&B disabled due to login timeout.
train:

# Cleaning image folders

Antes de pasar a la detección de materiales hay que borrar las augmented images y los .txt para crear los nuevos con los nuevos objetos.

In [ ]:
#Remove all augmented images
!rm /content/mate-detection/images/mates/*-aug*
!rm /content/mate-detection/labels/mates/*.txt

# Preprocessing materiales de mate labels
Preprocessing de labels para red de detección de materiales de mate.

In [ ]:
import os
import xml.etree.ElementTree as ET
%cd /content/yolov5
labels_dir = "../mate-detection/labels/mates/" # directorio que contiene los xml
img_dir = "../mate-detection/images/mates/"    # directorios con las imagenes
labels_materiales = ['plastico', 'calabaza', 'metal', 'madera']

# Procesar .xml a objetos
train_imgs = process_annotations(labels_dir, img_dir, labels_materiales)
# Crear .txt para cada objeto
create_txt_from_xml_labels(train_imgs, labels_materiales, 3)

/content/yolov5


# Create dataset yaml

Crear el .yaml usado por yolo para mate unicamente.
Se usa autosplit para dividir los datos en train/validation utilizando la regla 80/20

In [ ]:
import os
from utils.datasets import *;

autosplit(img_dir, (0.8, 0.2, 0))

with open("dataset.yaml", "w") as dataset:
    dataset.write("train: " + img_dir + "autosplit_train.txt" + os.linesep)
    dataset.write("val: " + img_dir + "autosplit_val.txt" + os.linesep)
    dataset.write("nc: " + str(len(labels_materiales)) + os.linesep)
    dataset.write("names: " + str(labels_materiales) + os.linesep)

100%|██████████| 1399/1399 [00:00<00:00, 36005.37it/s]

Autosplitting images from ../mate-detection/images/mates


# Train

Entrenamiento para detectar materiales de mate. Toma como base el archivo de pesos obtenido del paso anterior.

In [ ]:
!python train.py --img 416 --batch 32 --epochs 100 --cfg yolov5m.yaml --data dataset.yaml --weights 'mate_pesos.pt' --nosave --cache

github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v5.0-130-gfdbe527 torch 1.8.1+cu101 CUDA:0 (Tesla T4, 15109.75MB)

Namespace(adam=False, artifact_alias='latest', batch_size=32, bbox_interval=-1, bucket='', cache_images=True, cfg='./models/yolov5m.yaml', data='dataset.yaml', device='', entity=None, epochs=100, evolve=False, exist_ok=False, global_rank=-1, hyp='data/hyp.scratch.yaml', image_weights=False, img_size=[416, 416], label_smoothing=0.0, linear_lr=False, local_rank=-1, multi_scale=False, name='exp', noautoanchor=False, nosave=True, notest=False, project='runs/train', quad=False, rect=False, resume=False, save_dir='runs/train/exp', save_period=-1, single_cls=False, sync_bn=False, total_batch_size=32, upload_dataset=False, weights='mate_pesos.pt', workers=8, world_size=1)
tensorboard: Start with 'tensorboard --logdir runs/train', view at http://localhost:6006/
2021-06-01 03:12:58.311213: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Succe

# Cleaning train folders

In [ ]:
# Run this to clear detect and train folder

!rm -r ./runs/detect/*
!rm -r ./runs/train/*

rm: cannot remove './runs/detect/*': No such file or directory


# Prueba detect

In [ ]:
!python detect.py --weights '/content/yolov5/runs/train/exp/weights/last.pt' --img 416 --source '/content/mate-detection/test-sets/test-set-02/mate-calabaza-02.jpg'

detect: weights=['/content/yolov5/runs/train/exp/weights/last.pt'], source=/content/mate-detection/test-sets/test-set-02/mate-calabaza-02.jpg, data=data/coco128.yaml, imgsz=[416, 416], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-318-gc0380fd8 Python-3.10.12 torch-2.3.0+cu121 CUDA:0 (Tesla T4, 15102MiB)

Fusing layers... 
YOLOv5s summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
WARNING ⚠️ NMS time limit 0.550s exceeded
image 1/1 /content/mate-detection/test-sets/test-set-02/mate-calabaza-02.jpg: 224x416 1 mate, 46.1ms
Speed: 0.3ms pre-process, 46.1ms inference, 621.2ms NMS per image at shape (1, 3, 416, 416)
Results saved to runs/detect/